# GPU inference on Google Colab

Run project eval scripts on a **free T4 GPU** (~minutes instead of hours on Mac CPU).

**Before starting:** Runtime → Change runtime type → **T4 GPU**

See `colab/README.md` for how to upload your LoRA checkpoint.

In [ ]:
import torch
print("CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print(torch.cuda.get_device_name(0))

CUDA: False


In [ ]:
# Clone repo (public). Fork/private: paste your URL or upload the repo as a zip.
REPO_URL = "https://github.com/vladflorinfilip/Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography.git"

!rm -rf Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography
!git clone --depth 1 "{REPO_URL}"
%cd Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography

Cloning into 'Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography'...
remote: Enumerating objects: 157, done.
remote: Counting objects: 100% (157/157), done.
remote: Compressing objects: 100% (126/126), done.
remote: Total 157 (delta 35), reused 108 (delta 24), pack-reused 0 (from 0)
Receiving objects: 100% (157/157), 22.54 MiB | 3.73 MiB/s, done.
Resolving deltas: 100% (35/35), done.
/Users/vlad.filip/Desktop/Projects/Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography/colab/Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography


/Users/vlad.filip/Desktop/Projects/Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography/cot/lib/python3.9/site-packages/IPython/core/magics/osm.py:417: UserWarning: using dhist requires you to install the `pickleshare` library.
  self.shell.db['dhist'] = compress_dhist(dhist)[-100:]


In [ ]:
# Colab ships with torch; install the rest (no pin on torch).
!pip install -q peft "transformers<4.50" "datasets<4" accelerate openai pyyaml tqdm

You should consider upgrading via the '/Users/vlad.filip/Desktop/Projects/Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography/cot/bin/python3 -m pip install --upgrade pip' command.
You should consider upgrading via the '/Users/vlad.filip/Desktop/Projects/Poisoning-Chain-of-Though-Faithfulness-Through-Syntactic-Stenography/cot/bin/python3 -m pip install --upgrade pip' command.


In [ ]:
# --- Checkpoint: pick ONE method ---
!pip install google
USE_GITHUB_CHECKPOINT = False  # True only after you push checkpoints/qwen3b-cot-sft-v2 to GitHub
ADAPTER_DIR = "checkpoints/qwen3b-cot-sft-v2"

if USE_GITHUB_CHECKPOINT:
    print("Using checkpoint from cloned repo (must exist on GitHub).")
else:
    from google.colab import files
    import zipfile
    from pathlib import Path

    print("Upload qwen3b-cot-sft-v2.zip (create on Mac: see colab/README.md)")
    uploaded = files.upload()
    zpath = next(iter(uploaded))
    Path("checkpoints").mkdir(exist_ok=True)
    with zipfile.ZipFile(zpath, "r") as zf:
        zf.extractall("checkpoints")
    print("Extracted:", list(Path("checkpoints").iterdir()))

In [ ]:
# Optional: Hugging Face token if Qwen download fails (usually not needed).
# from google.colab import userdata
# import os
# os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")

from pathlib import Path
assert Path(ADAPTER_DIR, "adapter_model.safetensors").exists(), f"Missing {ADAPTER_DIR} — upload zip first"
print("Adapter OK:", ADAPTER_DIR)

In [ ]:
# Quick GPU smoke test (3 examples, ~30s on T4)
!python evaluation/evaluate_sbic.py \
  --model "{ADAPTER_DIR}" \
  --device cuda \
  --stance-judge regex \
  --limit 3 \
  --output /tmp/smoke.jsonl

!head -1 /tmp/smoke.jsonl | python -m json.tool | head -20

In [ ]:
# Full eval — change TASK: sbic | boolq | gsm8k | ethics
TASK = "sbic"
LIMIT = 100
STANCE_JUDGE = "regex"  # use "llm" if Azure secrets are set in Colab

!STANCE_JUDGE={STANCE_JUDGE} bash colab/run_gpu_eval.sh {TASK} {ADAPTER_DIR} {LIMIT}

In [ ]:
# Download JSONL results to your Mac
import shutil
from google.colab import files

shutil.make_archive("eval_outputs", "zip", "data/evaluation_data/qwen")
files.download("eval_outputs.zip")